In [ ]:
import torch
import torch.nn.functional as F

#ChatGPT generated, with some personal updates

def get_synthetic_batch(batch_size=4, seq_len=32, vocab_size=1000, seed=0):
    torch.manual_seed(seed)
    return {
        "input_ids": torch.randint(0, vocab_size, (batch_size, seq_len)),
        "attention_mask": torch.ones(batch_size, seq_len)
    }


def extract_outputs(model, batch):
    with torch.no_grad():
        outputs = model(
            **batch,
            output_attentions=True,   # IMPORTANT
            return_dict=True
        )

    hidden = outputs.last_hidden_state

    logits = getattr(outputs, "logits", None)
    attentions = getattr(outputs, "attentions", None)

    return hidden, logits, attentions


def assert_models_equivalent(
    model_a,
    model_b,
    atol_hidden=1e-5,
    atol_logits=1e-4,
    cosine_threshold=0.999,
    attention_cosine_threshold = 0.999,
    check_attention=False
):
    """
    Compares two models for functional equivalence.
    """

    model_a.eval()
    model_b.eval()

    batch = get_synthetic_batch()

    h_a, l_a, attn_a = extract_outputs(model_a, batch)
    h_b, l_b, attn_b = extract_outputs(model_b, batch)

    results = {}

    passed = True

    # -------------------------
    # 1. Hidden state check
    # -------------------------
    if h_a is not None and h_b is not None:
        hidden_mse = F.mse_loss(h_a, h_b).item()
        results["hidden_mse"] = hidden_mse

        if hidden_mse > atol_hidden:
            print(f"Hidden states differ too much: {hidden_mse:.6g}, with lower threshold: {atol_hidden}")
            passed = False

    # -------------------------
    # 2. Logit similarity
    # -------------------------
    if l_a is not None and l_b is not None:
        cos = F.cosine_similarity(
            l_a.flatten(),
            l_b.flatten(),
            dim=0
        ).item()

        results["logit_cosine"] = cos

        if cos < cosine_threshold:
            print(f"Logits not similar enough: {cos:.6g}, with upper threshold: {cosine_threshold}")
            passed = False
        
        logit_mse = F.mse_loss(l_a, l_b).item()
        results["logit_mse"] = logit_mse

        if logit_mse > atol_logits:
            print(f"Logits differ too much: {logit_mse:.6g}, with lower threshold: {atol_logits}")

    # -------------------------
    # 3. Attention check
    # -------------------------

    #TODO: adjust thi
    if check_attention and attn_a is not None and attn_b is not None:
        attn_diffs = []

        for a, b in zip(attn_a, attn_b):
            attn_diffs.append((a - b).abs().mean().item())

        attn_diff = sum(attn_diffs) / len(attn_diffs)
        results["raw_attention_diff"] = attn_diff

        a = torch.cat([x.flatten() for x in attn_a])
        b = torch.cat([x.flatten() for x in attn_b])

        attn_cos = F.cosine_similarity(a, b, dim=0).item()
        results["attention_cosine"] = attn_cos

        if attn_cos < attention_cosine_threshold:
            print(f"Attention not similar enough: {attn_cos}, with lower threshold: {attention_cosine_threshold}")
            passed = False

    if passed:
        print("Models are equivalent within tolerance")
    else:
        print("Models are NOT equivalent within tolerance")
        
    for k, v in results.items():
        print(f"{k}: {v:.6g}")

    return results

In [ ]:
import huggingface_hub as hb
from transformers import BertModel

model_identifier = "prajjwal1/bert-tiny"
model = BertModel.from_pretrained(model_identifier, attn_implementation="eager")



In [ ]:
from module_disentanglement import (disentangle_module, print_layers_with_params, test_disentanglement_form)

rank_fraction = 0.5
print_layers_with_params(model)
disentangled_model = disentangle_module(model, rank_fraction)
print()
print_layers_with_params(disentangled_model)

Save the disentangled model, only if it isn't already present in the config:

In [ ]:
import os
import json
from pathlib import Path

method = "disentanglement"
rank_fraction = rank_fraction
original_name = os.path.basename(model.name_or_path)

path = Path("../src/data/configs.json")

# load existing
if path.exists():
    with open(path, "r") as f:
        data = json.load(f)
else:
    data = []

exists = False 
for entry in data:
    if (entry["model"] == original_name 
        and entry["rank"] == rank_fraction 
        and entry["method"] == method):

        print("Same model is already saved under id:", entry["id"])
        exists = True
        break

if not exists:
    current_id = len(data)
    disentangled_name = "disentangled_model_id_" + str(current_id)

    # append new entry
    #TODO: add method of disentanglement used to config
    #Important note: currently only "fixed heuristic" is supported in "build_disentangled_structure"
    data.append({
        "id": current_id,
        "model": original_name,
        "rank": rank_fraction,
        "method": "disentanglement"
    })

    # write back
    with open(path, "w") as f:
        json.dump(data, f, indent=4)

    total_path = "../src/data/models/" + disentangled_name
    torch.save(disentangled_model.state_dict(), total_path)


Load models:

In [ ]:
from module_disentanglement import build_disentangled_structure
path_to_data = "../src/data/models/disentangled_model_id_0"
disentangled_model_reconstructed = build_disentangled_structure(model, rank_fraction)
state = torch.load(path_to_data)
disentangled_model_reconstructed.load_state_dict(state)

Test to what extent original and disentangled are equivalent: test should pass if rank_fraction = 1.0 is used. Important: assumes disentangled_model is the disentangled version of model.

In [ ]:

assert_models_equivalent(model, disentangled_model, check_attention=True)